# Notebook 3: full pipeline with the saved YOLO model -> fitted weight

This is the **deployable** pipeline. The lightweight YOLOv8-seg student trained and saved in **Notebook 2** replaces the large teacher. Everything else matches Notebook 1: measure every bird, **fit** the mass equation, and report weights from the fitted equation only.

$$\hat m \;=\; a\,\Big[\tfrac{\pi}{4}\,p_c^{3}\sum_{i\in S_\tau}\big(d_i^{(\delta)}\big)^2\,w_i\Big]^{\,b} \;=\; a\,V(\delta,\tau,n,c)^{\,b}$$

The equation is fitted **on the YOLO masks**, because they differ slightly from the teacher's masks and the parameters absorb that difference.

**Flow**

1. Load the FlockLabel export.
2. Load the saved YOLO model.
3. Follow one frame through the geometry.
4. Measure every bird with YOLO, and time it.
5. **Fit** the equation.
6. Weights from the fitted equation.
7. The most accurate density.
8. Comparison with the teacher (Notebook 1).
9. Videos with the fitted weight on each bird.
10. Files for the Raspberry Pi.

## 0. Install (run once, then restart the kernel)

In [ ]:
INSTALL = False                  # True = install / update broilerseg from GitHub, then restart
BRANCH = "Fitting"               # branch
SUBDIR = "broilerseg 4"          # folder in the repo that holds pyproject.toml ("" = pick from a list)

if INSTALL:
    import shutil, subprocess, sys
    from pathlib import Path
    REPO = "https://github.com/Akintanoreofe/BroilerWeightEstimation.git"
    SRC = Path.home() / "broilerseg_src"
    shutil.rmtree(SRC, ignore_errors=True)
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(SRC)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
    cands = sorted(p.parent.relative_to(SRC) for p in SRC.rglob("pyproject.toml")
                   if 'name = "broilerseg"' in p.read_text(errors="ignore"))
    print("broilerseg folders on the branch:", [str(c) for c in cands])
    pkg = SRC / SUBDIR if SUBDIR else SRC / cands[int(input(f"Pick 1-{len(cands)}: ")) - 1]
    if not (pkg / "pyproject.toml").is_file():
        raise FileNotFoundError(f"No pyproject.toml directly inside '{pkg.relative_to(SRC)}'")
    for cmd in ([sys.executable, "-m", "pip", "install", f"{pkg}[all]"],
                [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(pkg)]):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-2500:])
    print("Installed from", pkg.relative_to(SRC), "- now restart the kernel and set INSTALL = False")

## Settings

In [ ]:
# ---- data and models --------------------------------------------------------------
DRY_RUN = False                 # True = synthetic demo data + colour segmenter (no models)
MODEL_DIR = ""                  # folder saved by Notebook 2 ("" = ask); e.g. "~/broilerseg_models/broiler_yolov8n_seg"
MODEL_FORMAT = "pt"             # "pt", or "ncnn" / "onnx" to run exactly what the Pi runs
CONF = 0.25                     # YOLO confidence threshold
PEN_CROP = None                 # None = crop to the pen region saved with the model (if it was trained
                                # on pen crops); False = whole frames
TEACHER_RESULTS = "~/broilerseg_outputs/01_fitted_weight"   # Notebook 1 outputs, for comparison
MAX_FRAMES_PER_EVENT = 40       # labelled frames used per weighing event (None = all)
WEIGHT_UNIT = "g"               # "g" or "kg" (display only)
# ---- fitting: the values the search may choose from --------------------------------
OFFSETS = [-6, -4, -2, 0, 2, 4, 6]           # boundary offset (px)
TRIMS = [0.0, 0.03, 0.06, 0.10, 0.15]       # length trimmed at EACH end (fraction)
SLICES = [None, 5, 8, 10, 15, 20, 30]        # None = one slice per pixel column
RIDGE = 0.1                                 # pull of the exponent b towards 1 (0 = none)
FIX_B = None                                # None = decide from the data; True = force b = 1
NESTED_CV = True                            # unbiased error of the whole search (slower)
# ---- reference only ----------------------------------------------------------------
BASELINE_DENSITY = 900.0        # repository default, shown as the un-fitted baseline
SEED = 0
SAVE_FIGURES = True
OUTPUT_DIR = "~/broilerseg_outputs/03_yolo_weight"
# ------------------------------------------------------------------------------------

In [ ]:
import json
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import broilerseg
from broilerseg import viz
from broilerseg.calibration import calibrate_from_mask, summarize_calibrations
from broilerseg.config import PromptConfig, VolumeConfig
from broilerseg.evaluation import event_table, regression_metrics
from broilerseg.experiments import WeightExperiment
from broilerseg.io import load_flocklabel_export, subsample_per_event
from broilerseg.massmodel import (build_feature_table, build_scale_table, estimate_density,
                                  fit_mass_model, fitted_records)
from broilerseg.notebook import ask_for_directory, environment_report
from broilerseg.segmentation import Instance, clean_mask, select_bird

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
viz.set_style()
viz.set_weight_unit(WEIGHT_UNIT)
pd.set_option("display.max_columns", 40)
OUT = Path(OUTPUT_DIR).expanduser().resolve(); (OUT / "figures").mkdir(parents=True, exist_ok=True)
F = viz.weight_factor()
FIG = 0
def show(fig, name):
    # Display a figure and save it as a numbered PNG.
    global FIG
    FIG += 1
    if SAVE_FIGURES:
        viz.save(fig, OUT / "figures" / f"{FIG:02d}_{name}.png")
    plt.show()
def wmetrics(actual_kg, pred_kg):
    # Error metrics in the display unit (MAPE and R^2 are unit-free).
    return {k: round(v, 3 if WEIGHT_UNIT == "kg" else 1)
            for k, v in regression_metrics(actual_kg * F, pred_kg * F).items()}

print("broilerseg", broilerseg.__version__)
if broilerseg.__version__ < "0.5.0":
    print("This notebook needs broilerseg >= 0.4.0 - set INSTALL = True above, run it, restart.")
display(pd.Series(environment_report(), name="environment").to_frame())

## 1. The FlockLabel export (scale readings)

Paste the folder that FlockLabel's **Export for evaluation** created (it has `test_videos/` and `ground_truth/`). A dataset export (with `manifest.csv`) also works. To skip the prompt, set the `BROILERSEG_WEIGHT_DIR` environment variable.

In [ ]:
if DRY_RUN:
    from broilerseg.synthetic import make_demo_data
    demo = make_demo_data(OUT / "demo_data", n_events=8, frames_per_event=10)
    WEIGHT_DIR = demo["export"]
    print("DRY RUN: synthetic export at", WEIGHT_DIR)
else:
    WEIGHT_DIR = ask_for_directory("Paste the FlockLabel export folder (weight validation data)",
                                   env_var="BROILERSEG_WEIGHT_DIR")
export = load_flocklabel_export(WEIGHT_DIR)
print(export)
display(export.events.head(10))
show(viz.plot_weight_dataset_overview(export.samples), "dataset_overview")
samples = subsample_per_event(export.samples, MAX_FRAMES_PER_EVENT, SEED)
ev_w = samples.groupby("event_id").weight_kg.median()
print(f"Using {len(samples)} frames from {samples.event_id.nunique()} events; event weights "
      f"{ev_w.min() * F:.0f}-{ev_w.max() * F:.0f} {WEIGHT_UNIT} "
      f"(variation {100 * ev_w.std(ddof=0) / ev_w.mean():.1f}%)")
if samples.event_id.nunique() < 3:
    print("WARNING: fewer than 3 events - cross-validation needs more weighing events.")

## 2. Load the saved YOLO model (from Notebook 2)

Paste the model folder that Notebook 2 printed at the end, or set `MODEL_DIR` above. The model card shows how the model was trained and how well it matched the teacher.

In [ ]:
from broilerseg.distill import load_student
if not MODEL_DIR and not DRY_RUN:
    MODEL_DIR = str(ask_for_directory("Paste the saved YOLO model folder from Notebook 2",
                                      env_var="BROILERSEG_MODEL_DIR"))
card = {}
model_path = Path(MODEL_DIR).expanduser() if MODEL_DIR else None
if model_path is not None and (model_path / "model_card.json").is_file():
    card = json.loads((model_path / "model_card.json").read_text())
    display(pd.Series({k: v for k, v in card.items() if not isinstance(v, (list, dict))},
                      name="model card").to_frame())
    if card.get("metrics"):
        show(viz.plot_student_metrics(card["metrics"]), "student_metrics_from_card")
    if (model_path / "results.csv").is_file():
        from broilerseg.distill import load_training_curves
        show(viz.plot_training_curves(load_training_curves(model_path)), "training_curves")
use_stand_in = DRY_RUN or card.get("stand_in", False) or not (model_path and (model_path / "best.pt").exists()
                                                              or MODEL_FORMAT != "pt")
if use_stand_in:
    from broilerseg.synthetic import ColorSegmenter
    segmenter = ColorSegmenter(); segmenter.name = "stand-in for YOLO (dry run / no trained weights)"
    print(segmenter.name)
else:
    segmenter, card = load_student(model_path, MODEL_FORMAT, conf=CONF, pen_crop=PEN_CROP)
    print("Loaded", segmenter.name, "at imgsz", segmenter.imgsz)
    if getattr(segmenter, "roi", None) is not None:
        r = segmenter.roi
        print(f"Two-stage: frames are cropped to the saved pen region "
              f"(x {r.fx0:.2f}-{r.fx1:.2f}, y {r.fy0:.2f}-{r.fy1:.2f} of the frame) before YOLO runs")

## 3. One frame through the geometry

This shows what YOLO sees, how the feeder sets the scale, which bird is weighed, and how its mask is sliced. No weight is computed yet: weights come from the fitted equation in section 6.

In [ ]:
EXAMPLE = 0
exp_one = WeightExperiment(segmenter, VolumeConfig(BASELINE_DENSITY, None), "enclosing_circle", "frame")
exp_one.run(samples.iloc[[EXAMPLE]])
sid = int(samples.sample_id.iloc[EXAMPLE]); s = samples.iloc[EXAMPLE]
image, result = exp_one.image(sid), exp_one.results[sid]
print(f"{s.video_name} frame {s.frame_label}: scale {viz.fmt_weight(s.weight_kg)} | "
      f"{segmenter.name}: {result.seconds:.2f} s | found {result.counts()}")
show(viz.plot_segmentation_panels(image, result, f"{s.video_name} frame {s.frame_label}"), "example_panels")
feeder = result.best("feeder")
cal = calibrate_from_mask(feeder.mask if feeder else None, method="enclosing_circle")
show(viz.plot_feeder_calibration(image, cal, feeder.mask if feeder else None), "example_calibration")
bird, info = select_bird(result, (s.x1, s.y1, s.x2, s.y2))
show(viz.plot_bird_selection(image, result, bird, (s.x1, s.y1, s.x2, s.y2), info), "example_selection")
if sid in exp_one.estimates:
    est = exp_one.estimates[sid]
    show(viz.plot_alignment(exp_one.birds[sid].mask, est), "example_alignment")
    show(viz.plot_slices(est), "example_slices")

In [ ]:
# The pen crop YOLO actually sees for this frame
if getattr(segmenter, "last_roi", None) is not None:
    roi = segmenter.last_roi
    crop = roi.crop(image, getattr(segmenter, "mask_outside", False))
    show(viz.plot_two_stage(image, None, roi, crop, roi.cut(result),
                            title="Pen crop used by the YOLO student"), "pen_crop")

## 4. Measure every bird (with YOLO)

YOLO runs once per frame; results are cached, so nothing below re-segments. This stage produces the **inputs** of the fit: one cleaned bird mask and one feeder measurement per frame.

In [ ]:
experiment = WeightExperiment(segmenter, VolumeConfig(BASELINE_DENSITY, None),
                              "enclosing_circle", "video_median")
records = experiment.run(samples)
print(f"{len(experiment.birds)} / {len(records)} frames have a bird mask")
show(viz.plot_status_breakdown(records.assign(pred_kg=records.volume_cm3)), "status")
sec = records.seg_seconds.dropna()
print(f"YOLO: {1000 * sec.mean():.0f} ms per frame on this computer ({1 / sec.mean():.1f} fps)")
rng = np.random.default_rng(SEED)
pick = rng.choice(records.sample_id, size=min(8, len(records)), replace=False)
meta = records.set_index("sample_id")
show(viz.plot_gallery([(experiment.image(int(i)), experiment.results[int(i)],
                        f"{meta.loc[i, 'video_name'][:18]} f{meta.loc[i, 'frame_label']}") for i in pick],
                      ncols=4, title=f"{segmenter.name} masks"), "gallery")
show(viz.plot_calibration_stability(records), "calibration_stability")
show(viz.plot_calibration_methods([experiment.calibrations[int(i)] for i in records.sample_id]),
     "calibration_methods")

## 5. Fit the mass equation

Every combination of $\delta, \tau, n$, calibration method and scope is tried, with $b$ fixed at 1 or fitted. For each one, $a$ and $b$ have an exact least-squares solution. Each combination is scored by **leave-one-event-out** cross-validation: every event is predicted by a model that never saw it. **Nested** cross-validation then repeats the whole search without each event. That is the honest error of the complete procedure, and the number to report.

In [ ]:
features = build_feature_table(experiment, OFFSETS, TRIMS, SLICES)
scales = build_scale_table(experiment)
report = fit_mass_model(features, scales, fix_b=FIX_B, ridge=RIDGE,
                        baseline_density=BASELINE_DENSITY, nested=NESTED_CV)
model = report.model
print("FITTED EQUATION\n  " + model.equation(WEIGHT_UNIT))
print(f"\n{report.n_events} events, weight variation {100 * report.weight_cv:.1f}% -> exponent b: "
      f"{report.b_status()}")
display(report.summary().to_frame("value"))
comp = report.comparison.copy()
for c in ("rmse_kg", "mae_kg", "bias_kg"):
    comp[c.replace("_kg", f"_{WEIGHT_UNIT}")] = comp.pop(c) * F
display(comp.round(2))
report.cv_table.to_csv(OUT / "cv_table.csv", index=False)

### 5a. The fitted equation and what each part buys

In [ ]:
show(viz.plot_mass_fit(report), "fit")
show(viz.plot_model_comparison(report), "model_comparison")

### 5b. Which settings matter, and are they stable?
Each bar shows the best cross-validated error reachable with that value of a setting. A flat panel means the setting hardly matters for your data. The nested table shows the setting chosen when each event is held out: different choices in different rows mean the data can't pin that setting down yet.

In [ ]:
show(viz.plot_parameter_sensitivity(report), "sensitivity")
show(viz.plot_cv_landscape(report, top=25), "top_configs")
show(viz.plot_nested_choices(report), "nested_cv")
show(viz.plot_fold_stability(report), "fold_stability")
mid = report.data.iloc[(report.data.volume_cm3 - report.data.volume_cm3.median()).abs().argsort()[:1]]
show(viz.plot_mask_adjustments(experiment.birds[int(mid.sample_id.iloc[0])].mask, model), "mask_adjustments")

## 6. Weights from the fitted equation

Every weight below comes from the fitted equation. The accuracy plots use **out-of-fold** estimates: each event is predicted by the equation fitted on the *other* events, so the accuracy is not flattered. `pred_final_kg` in the saved table uses the final equation fitted on all events; use that one on new data.

In [ ]:
fitted = fitted_records(report, records)
fitted.to_csv(OUT / "records_fitted.csv", index=False)
print(f"Frame level ({WEIGHT_UNIT}):", wmetrics(fitted.actual_kg, fitted.pred_kg))
ev = event_table(fitted)
print(f"Event level ({WEIGHT_UNIT}):", wmetrics(ev.actual_kg, ev.pred_kg))
display(viz.weight_table(ev).round(1))
show(viz.plot_accuracy_overview(fitted, "Fitted equation (out-of-fold)"), "accuracy")
show(viz.plot_per_event(fitted, "Fitted equation: every weighing event"), "per_event")
show(viz.plot_frame_traces(fitted), "frame_traces")
show(viz.plot_error_analysis(fitted), "error_analysis")
show(viz.plot_implied_density(fitted, model.effective_density() if model.b == 1 else BASELINE_DENSITY),
     "implied_density")

### 6a. Best and worst frames, end to end (fitted equation)

In [ ]:
ok = fitted.dropna(subset=["pred_kg"]).assign(abs_pct=lambda d: d.pct_error.abs()).sort_values("abs_pct")
for tag, rows in (("BEST", ok.head(2)), ("WORST", ok.tail(3))):
    for _, r in rows.iterrows():
        i = int(r.sample_id)
        bird_i = experiment.birds[i]
        est = model.estimate(bird_i.mask, r.pixel_to_cm)
        show(viz.plot_volume_storyboard(experiment.image(i), experiment.results[i], bird_i, est,
                                        r.actual_kg, f"{tag}: {r.video_name} frame {r.frame_label} "
                                        f"(out-of-fold {r.pct_error:+.1f}%; numbers panel = final equation)"),
             f"storyboard_{tag.lower()}_{i}")

## 7. The most accurate density

With $b = 1$, the equation is the physical model $\hat m = \rho\,V\cdot10^{-6}$. The density is fitted on all events with the settings chosen in section 5. Its uncertainty is shown as a bootstrap 95% interval (resampling whole events) and as the range of values obtained when each event is left out. The density is only valid together with those settings.

In [ ]:
dens = estimate_density(report, n_boot=2000, seed=SEED)
print(f"Most accurate density: {dens['density_kg_m3']:.0f} kg/m^3 "
      f"(95% CI {dens['ci95'][0]:.0f}-{dens['ci95'][1]:.0f}; leave-one-event-out "
      f"{dens['loeo_min']:.0f}-{dens['loeo_max']:.0f})")
show(viz.plot_density_estimate(dens), "density")
(OUT / "mass_model.json").write_text(json.dumps(
    {**model.as_dict(), "best_density_kg_m3": dens["density_kg_m3"], "density_ci95": dens["ci95"],
     "cv_rmse_kg": float(report.cv_table.cv_rmse_kg.iloc[0]),
     "nested_cv_rmse_kg": float(report.comparison.set_index("model").rmse_kg.iloc[-1]),
     "events": report.n_events, "segmenter": segmenter.name,
     "model_dir": str(model_path) if model_path else None}, indent=2, default=str))
print("Saved", OUT / "mass_model.json")

## 8. Student vs teacher

When Notebook 1 has been run (its outputs are in `TEACHER_RESULTS`), the YOLO pipeline is compared with the teacher pipeline on the same weighing events. Both use their own fitted equation, and both errors are out-of-fold.

In [ ]:
T = Path(TEACHER_RESULTS).expanduser()
if (T / "records_fitted.csv").is_file():
    teacher_rec = pd.read_csv(T / "records_fitted.csv")
    teacher_rec["event_id"] = teacher_rec.event_id.astype(str)
    common = set(teacher_rec.sample_id) & set(fitted.sample_id)
    runs = {"teacher (Notebook 1)": teacher_rec[teacher_rec.sample_id.isin(common)],
            f"YOLO student": fitted[fitted.sample_id.isin(common)]}
    print(f"{len(common)} frames in both")
    show(viz.plot_method_comparison(runs, "Weight accuracy: teacher masks vs YOLO masks"), "teacher_vs_student")
    eq = {"teacher (Notebook 1)": json.loads((T / "mass_model.json").read_text())
          if (T / "mass_model.json").is_file() else {},
          "YOLO student": json.loads((OUT / "mass_model.json").read_text())}
    display(pd.DataFrame(eq).loc[["a", "b", "offset_px", "trim", "slice_count", "calib_method",
                                  "calib_scope", "best_density_kg_m3", "nested_cv_rmse_kg"]])
    both = teacher_rec[["sample_id", "pred_kg"]].merge(fitted[["sample_id", "pred_kg"]], on="sample_id",
                                                       suffixes=("_teacher", "_student"))
    show(viz.plot_pred_vs_actual(both, pred="pred_kg_student", actual="pred_kg_teacher",
                                 title="Frame by frame: YOLO estimate vs teacher estimate",
                                 xlabel=f"teacher estimate ({WEIGHT_UNIT})",
                                 ylabel=f"YOLO estimate ({WEIGHT_UNIT})"), "frame_agreement")
else:
    print(f"No Notebook 1 results in {T} - run Notebook 1 (or set TEACHER_RESULTS) to compare.")

## 9. Videos: YOLO + the fitted weight on each broiler

Each video is segmented, and the weighed broiler is boxed with its weight from the **final fitted equation** above it. The scale reading and % error are shown when FlockLabel has them. The pixel scale uses the fitted calibration method and scope.

In [ ]:
# ---- settings -------------------------------------------------------------------
N_VIDEOS = 6           # videos from the export
VIDEO_NAMES = None     # or a list of names
STRIDE = 2             # segment every Nth frame
MAX_FRAMES = 120       # per video
OUT_WIDTH = 640
GRID_COLUMNS = 3
# ----------------------------------------------------------------------------------
import base64, shutil, subprocess
from IPython.display import HTML
from broilerseg.io import iter_frames, video_info

vid_dir = OUT / "videos"; vid_dir.mkdir(parents=True, exist_ok=True)
FFMPEG = shutil.which("ffmpeg")
camera_px = float(fitted.pixel_to_cm.median())          # fitted data's scale (camera scope)

def draw_label(img, box, lines, color=(255, 214, 0)):
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, max(2, img.shape[1] // 320))
    scales_ = [0.75 * img.shape[1] / 640, 0.5 * img.shape[1] / 640]
    sizes = [cv2.getTextSize(t, cv2.FONT_HERSHEY_SIMPLEX, sc, 2)[0] for t, sc in zip(lines, scales_)]
    pad = 4; h = sum(sz[1] + 2 * pad for sz in sizes); w = max(sz[0] for sz in sizes) + 2 * pad
    top = y1 - h if y1 - h >= 0 else min(y2, img.shape[0] - h); left = min(max(0, x1), img.shape[1] - w)
    cv2.rectangle(img, (left, top), (left + w, top + h), color, -1)
    y = top
    for t, sc, sz in zip(lines, scales_, sizes):
        y += sz[1] + pad
        cv2.putText(img, t, (left + pad, y), cv2.FONT_HERSHEY_SIMPLEX, sc, (0, 0, 0), 2, cv2.LINE_AA)
        y += pad

def write_video(images, path, fps):
    size = (images[0].shape[1], images[0].shape[0])
    for codec in ("avc1", "mp4v"):
        wr = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*codec), fps, size)
        if wr.isOpened():
            break
    for im in images:
        wr.write(cv2.cvtColor(im, cv2.COLOR_RGB2BGR))
    wr.release()
    if codec != "avc1" and FFMPEG:
        tmp = path.with_name(path.stem + "_h264.mp4")
        if subprocess.run([FFMPEG, "-y", "-loglevel", "error", "-i", str(path), "-c:v", "libx264",
                           "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(tmp)]).returncode == 0:
            tmp.replace(path); codec = "avc1"
    return codec == "avc1"

def process_video(video_path, gt):
    info = video_info(video_path)
    gt_by_frame = {int(r.frame_index0): r for _, r in gt.iterrows()}
    sc = OUT_WIDTH / info["width"]; size = (OUT_WIDTH, int(round(info["height"] * sc)))
    frames = []
    for idx, rgb in iter_frames(video_path, indices=list(range(0, info["n_frames"], STRIDE))[:MAX_FRAMES]):
        res = segmenter.segment(rgb)
        feeder = res.best("feeder")
        cal = calibrate_from_mask(feeder.mask if feeder else None, method=model.calib_method)
        g = gt_by_frame.get(idx)
        bird, sel = select_bird(res, (g.x1, g.y1, g.x2, g.y2) if g is not None else None)
        crop = box = None
        if bird is not None:
            m = clean_mask(bird.mask); ys, xs = np.nonzero(m); pad = abs(model.offset_px) + 3
            y0, x0 = max(0, ys.min() - pad), max(0, xs.min() - pad)
            crop = m[y0:ys.max() + pad + 1, x0:xs.max() + pad + 1]
            box = (xs.min() * sc, ys.min() * sc, (xs.max() + 1) * sc, (ys.max() + 1) * sc)
        frames.append({"idx": idx, "img": cv2.resize(viz.blend_masks(rgb, res, alpha=0.3, contours=False),
                                                     size, interpolation=cv2.INTER_AREA),
                       "px": cal.pixel_to_cm, "gt": g, "crop": crop, "box": box, "reason": sel["reason"]})
        del rgb, res
    px = np.array([f["px"] for f in frames], float)
    p_video = float(np.nanmedian(px)) if np.isfinite(px).any() else np.nan
    rows, out_imgs = [], []
    for f in frames:
        p = {"frame": f["px"], "video": p_video, "camera": camera_px}[model.calib_scope]
        img = f["img"].copy(); actual = f["gt"].weight_kg if f["gt"] is not None else np.nan
        pred = model.predict(f["crop"], p) if f["crop"] is not None and np.isfinite(p) else np.nan
        if np.isfinite(pred):
            lines = [viz.fmt_weight(pred)]
            if np.isfinite(actual):
                lines.append(f"scale {viz.fmt_weight(actual)} ({100 * (pred - actual) / actual:+.1f}%)")
            draw_label(img, f["box"], lines)
        else:
            cv2.putText(img, f"no estimate: {f['reason']}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (255, 80, 80), 2, cv2.LINE_AA)
        out_imgs.append(img)
        rows.append({"video": video_path.name, "frame": f["idx"], "pred_kg": pred, "actual_kg": actual})
    out = vid_dir / f"{video_path.stem}_yolo_fitted.mp4"
    return out, pd.DataFrame(rows), write_video(out_imgs, out, max(1.0, info["fps"] / STRIDE))

names = VIDEO_NAMES or list(dict.fromkeys(export.samples.video_name))[:N_VIDEOS]
cards, table, playable = [], [], True
for k, n in enumerate(names, 1):
    gt = export.samples[export.samples.video_name == n]
    print(f"[{k}/{len(names)}] {n} ...", flush=True)
    out, df, ok_play = process_video(Path(gt.file.iloc[0]), gt)
    playable &= ok_play
    okp = df.dropna(subset=["pred_kg"]); est_m, act_m = okp.pred_kg.median(), df.actual_kg.median()
    cap = f"<b>{n}</b><br>estimate {viz.fmt_weight(est_m) if np.isfinite(est_m) else '-'}"
    if np.isfinite(act_m) and np.isfinite(est_m):
        cap += f" | scale {viz.fmt_weight(act_m)} | <b>{100 * (est_m - act_m) / act_m:+.1f}%</b>"
    data = base64.b64encode(out.read_bytes()).decode()
    cards.append(f'<div><video src="data:video/mp4;base64,{data}" autoplay loop muted controls playsinline '
                 f'style="width:100%;border-radius:6px"></video><div style="font:12px sans-serif">{cap}</div></div>')
    table.append({"video": n, f"median estimate ({WEIGHT_UNIT})": est_m * F,
                  f"median scale ({WEIGHT_UNIT})": act_m * F,
                  "error %": 100 * (est_m - act_m) / act_m if np.isfinite(act_m) else np.nan})
display(HTML(f'<div style="display:grid;grid-template-columns:repeat({GRID_COLUMNS},1fr);gap:12px">'
             f'{"".join(cards)}</div>'))
if not playable:
    print("Some videos are not H.264 and may not play here: conda install -c conda-forge ffmpeg, then re-run.")
display(pd.DataFrame(table).round(1))

## 10. Files for the Raspberry Pi

The fitted equation is copied into the model folder, so a single folder holds everything the Pi needs: the YOLO model (`best_ncnn_model/`) and the equation (`mass_model.json`).

In [ ]:
if model_path is not None and model_path.is_dir():
    import shutil
    shutil.copy2(OUT / "mass_model.json", model_path / "mass_model.json")
    print("Copied the fitted equation to", model_path / "mass_model.json")
print(json.dumps(json.loads((OUT / "mass_model.json").read_text()), indent=2)[:1500])

**On the Pi**

```python
import json, cv2
from broilerseg.distill import load_student
from broilerseg.massmodel import MassModel
from broilerseg.calibration import calibrate_from_mask
from broilerseg.segmentation import select_bird, clean_mask

seg, card = load_student("broiler_yolov8n_seg", fmt="ncnn")   # crops to the saved pen region itself
eq = json.load(open("broiler_yolov8n_seg/mass_model.json"))
model = MassModel(eq["a"], eq["b"], eq["offset_px"], eq["trim"],
                  None if eq["slice_count"] == "per-column" else int(eq["slice_count"]),
                  eq["calib_method"], eq["calib_scope"])
res = seg.segment(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
p = calibrate_from_mask(res.best("feeder").mask, method=model.calib_method).pixel_to_cm
bird, _ = select_bird(res)
grams = 1000 * model.predict(clean_mask(bird.mask), p)
```

**Reading the results**

* Report the **nested** cross-validated error (last row of the comparison in section 5).
* If YOLO's error is clearly worse than the teacher's (section 8), the student needs more or better training frames. Go back to Notebook 2.